Testing Eval with huggingface text-generation models

Using Langsmith

In [ ]:
from ragas.integrations.langchain import EvaluatorChain 
from ragas.metrics import faithfulness, context_precision
# from ragas.metrics.collections import faithfulness, context_precision
from langsmith.evaluation import LangChainStringEvaluator
from langchain_community.embeddings import HuggingFaceEmbeddings

In [5]:
from ..core.generation.llm import HUGGINGFACE
from ..core.utils.config import huggingface_config

resource = huggingface_config()

# llm
provider = HUGGINGFACE()
provider.model_name = resource['evaluation model']
provider.task = 'text-generation'
provider.load_model()
llm = provider.pipeline

# embedding model
embedding_model = HuggingFaceEmbeddings(model_name=resource['embedding model'])


ImportError: attempted relative import with no known parent package

Evaluation

Faithfullness

In [ ]:
faithfulness_chain = EvaluatorChain(
    metric = faithfulness, # the metric to use
    llm = llm, # or any other model
    embedding = embedding_model, # or any other embedding model
)

eval_result = faithfulness_chain(
    {
        "question": "What is the capital of France?",# user question
        "answer": "The capital of France is Paris.", # generated answer from the model
        "context": ["France is a country in Europe. Its capital city is Paris, known for its art, fashion, and culture."], # doc chunks available to the model
    }
)

print(eval_result)



using openeval

In [ ]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

judge = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key="",
)
anthropic_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge = judge
    # judge=ChatAnthropic(model="claude-3-5-sonnet-latest", temperature=0.5),
)

In [7]:
inputs = "How is the weather in San Francisco?"
# These are fake outputs, in reality you would run your LLM-based system to get real outputs
outputs = "Thanks for asking! The current weather in San Francisco is sunny and 90 degrees."
# When calling an LLM-as-judge evaluator, parameters are formatted directly into the prompt
reference = "The weather in San Francisco is currently sunny and 90 degrees."
eval_result = anthropic_evaluator(
    inputs=inputs,
    outputs=outputs,
    reference_outputs=reference,


)

print(eval_result)

{'key': 'score', 'score': True, 'comment': 'The model output accurately and completely answers the question about the weather in San Francisco. It states that the weather is "sunny and 90 degrees," which aligns with the provided reference output. There are no factual errors, logical inconsistencies, or missing key information. The conversational opening "Thanks for asking!" does not detract from the correctness of the weather information provided. Thus, the score should be: true.', 'metadata': None}


Experimenting with store data function

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from typing import List

def export_to_csv(path: str, filename: str, data: List[str]):
    """stores interaction data"""

    # ensuring that the main folder for storage exixts before proceeding
    folder = Path(path)
    if not folder.exists():
        raise ValueError('folder does not exist: copy the absolute path')
    try:
        # check if file exists or not inorder to prevent duplicate headers
        file = filename + ".csv"
        complete_file = Path(folder / file)
        file_exists = complete_file.exists()

        # convert the file to the preferred format and export for evaluation (.csv)
        interaction_data = np.array(data).reshape((1,3))
        df = pd.DataFrame(interaction_data,columns=['input','output','reference'])
        
        df.to_csv(complete_file, mode='a', header=not file_exists, index=False)
    except ValueError:
        raise ValueError('check the arguments')


check(
    r'C:\Users\Omolayo-Akinola\Documents\projects\engineering\RAG chatbot\data\raw',
    'test',
    ['test1','test2','test3']
)


check(
    r'C:\Users\Omolayo-Akinola\Documents\projects\engineering\RAG chatbot\data\raw',
    'test',
    ['test4','test5','test6']
)

Creating vector index

In [1]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
client = Pinecone(api_key=os.environ.get('PINECONE_API_KEY'))

index = client.create_index(
    name = os.environ.get('PINECONE_INDEX').lower(),
    dimension = 384,
    spec = ServerlessSpec(
        cloud = os.environ.get('PINECONE_CLOUD_PROVIDER'),
        region = os.environ.get('PINECONE_REGION'),
    ),
    metric = 'cosine',
    )


Testing models

In [23]:
from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cpu


In [2]:
pipe("The quick brown fox jumps over the lazy dog. The dog was not happy about it and decided to chase the fox around the block.")

Your max_length is set to 142, but your input_length is only 29. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)


[{'summary_text': 'The quick brown fox jumps over the lazy dog. The dog was not happy about it and decided to chase the fox around the block. The fox ran off and the dog chased after it for a few more minutes. It was not the first time the fox had jumped over the dog.'}]

In [3]:
pipe("The quick brown fox jumps over the lazy dog. The dog was not happy about it and decided to chase the fox around the block.", max_length=20, min_length=5, do_sample=False)

[{'summary_text': 'The quick brown fox jumps over the lazy dog. The dog was not happy about it'}]

understanding vector db return type

In [100]:
import os 
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_community.embeddings import HuggingFaceEmbeddings
load_dotenv()

client = Pinecone(api_key=os.environ.get('PINECONE_API_KEY'))
index = client.Index(os.environ.get('PINECONE_INDEX').lower())

# query = "how do i get a refund?"
# query = "what is MotoCura’s warranty and return policy"
query = "what is motocura about? and what do they offer?"
embed_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2' )
query_embeddings = embed_model.embed_query(query)




In [101]:
results = index.query(
    vector=query_embeddings,
    top_k=7,
    include_metadata=True,
    namespace='chatbot'
)

In [ ]:
result = ''
for each in results['matches']:
    # print(each['metadata']['text'])
    result += each['metadata']['text']
    final_result = ''.join(result)
print(results['matches'])
che[doc for doc in results if doc is not None]
print(final_result)

QueryResponse(matches=[ScoredVector(id='002fdbcc-605e-455e-9e78-adfec2e2d473-19', score=0.826482773, values=[], metadata={'source': 'motocura chat docs', 'text': 'General Questions \nQ1: What is MotoCura?'}), ScoredVector(id='b69ad464-ff3d-4d32-a58b-fbbd41a04c7a-1', score=0.818883419, values=[], metadata={'source': 'motocura chat docs', 'text': 'Company Overview & Mission Statement \nOverview \nMotoCura is a  tech startup transforming automotive logistics and mobility across Africa.'}), ScoredVector(id='9d81566d-446b-4297-8732-43a82780b755-13', score=0.784836829, values=[], metadata={'source': 'motocura chat docs', 'text': 'Terms & Conditions \nIntroduction \nWelcome to MotoCura — a tech-driven platform providing automotive services such as car \nrentals, repairs, and vehicle sales.'}), ScoredVector(id='002fdbcc-605e-455e-9e78-adfec2e2d473-20', score=0.776316643, values=[], metadata={'source': 'motocura chat docs', 'text': 'MotoCura is a tech startup that makes car ownership and mobili

In [104]:
pipe(final_result, max_length=100, min_length=5, do_sample=False)[0]['summary_text']

'MotoCura is a tech startup transforming automotive logistics and mobility across Africa. MotoCura combines technology, ethics, and sustainability.'

In [108]:
print("nigga\n nigga")

nigga
 nigga


In [111]:
from pathlib import Path

print(Path(__file__).resolve().parents[2])

if (Path.home() / "scripts").exists():
    print('True')
else:
    print("false")

NameError: name '__file__' is not defined

In [1]:
from huggingface_hub import InferenceClient
import os
from dotenv import load_dotenv
load_dotenv()
client = InferenceClient(
    model="sentence-transformers/all-MiniLM-L6-v2",
    api_key=os.environ['HUGGINGFACE_API_KEY']
)

# extract embeddings
query = 'how do i fly a kite?'
embeds = client.feature_extraction(query)

print(embeds)

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ 4.52510118e-02  3.89183946e-02 -5.02807461e-02  2.56264322e-02
 -6.89440072e-02  6.30438887e-03  5.69343269e-02 -4.69480418e-02
 -2.14829110e-02  2.96337586e-02  5.51114194e-02 -3.45699452e-02
 -1.09385349e-01 -2.41638869e-02  2.84242369e-02  8.60583931e-02
 -6.16027080e-02 -1.59530656e-03 -3.43934521e-02  4.34880843e-03
  2.00010352e-02  3.37217078e-02  4.14763624e-03  6.25311211e-03
 -2.14437507e-02  2.30823923e-03 -6.01170920e-02  2.52081864e-02
  1.58393271e-02 -7.29579059e-03  4.31772992e-02  3.86562459e-02
 -7.70204514e-02  1.88666154e-02 -9.79418084e-02  1.05888275e-02
  2.76037026e-02 -3.26500796e-02 -1.31091252e-02  1.01357047e-03
  6.93175718e-02 -7.42417062e-04  5.41850599e-03  3.86349931e-02
 -1.52815264e-02  8.44763666e-02  8.26590732e-02  3.30604315e-02
  1.16756335e-01 -4.54549352e-03 -5.81129417e-02 -4.91666608e-02
 -1.40314000e-02 -5.35869673e-02  9.04012769e-02  3.79182026e-02
 -4.99320729e-03 -1.03025977e-02  8.12771078e-03 -4.92920354e-02
 -9.93553270e-03 -1.35999